# Qwen3 LoRA 微调：城市治理回复助手 (U-Gov)

目标：用免费官方回复数据把 `Qwen/Qwen3-0.6B`（可换 1.5B/3B）微调成政务回复助手，导出可供 Ollama 使用的 merged 模型。

**使用前**：
1. 打开右侧 Settings：Accelerator = **GPU**、Internet = **On**。
2. 左上 Add Input → Upload Dataset，把本机的 `data/lora/train.jsonl` 和 `eval.jsonl` 传上去（会自动被找到）。
3. **Run All**。

详细步骤见 `docs/KAGGLE_FINETUNE.md`。

In [ ]:
!pip install -q "transformers>=4.45" peft accelerate datasets bitsandbytes
print("deps ok")

In [ ]:
import glob, json, os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

BASE_MODEL = "Qwen/Qwen3-0.6B"   # 显存够可换 Qwen/Qwen3-1.5B / Qwen3-3B
OUT_DIR = "/kaggle/working/ugov-qwen"
os.makedirs(OUT_DIR, exist_ok=True)

train_file = glob.glob("/kaggle/input/*/train.jsonl")
eval_file  = glob.glob("/kaggle/input/*/eval.jsonl")
assert train_file and eval_file, "未在 /kaggle/input 找到 train.jsonl / eval.jsonl，请先上传 dataset"
print("train:", train_file[0])
print("eval :", eval_file[0])

In [ ]:
from torch.utils.data import Dataset

def load(path):
    out = []
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if line: out.append(json.loads(line))
    return out

# 只用 assistant 回复算损失（屏蔽 system/user 提示词）
class SFT(Dataset):
    def __init__(self, samples, tok, max_len=512):
        self.data = []
        for m in samples:
            full = tok(tok.apply_chat_template(m["messages"], tokenize=False, add_generation_prompt=False),
                       truncation=True, max_length=max_len)
            prom = tok(tok.apply_chat_template(m["messages"], tokenize=False, add_generation_prompt=True),
                       truncation=True, max_length=max_len)
            plen = len(prom["input_ids"])
            ids  = full["input_ids"]
            if plen >= len(ids): plen = len(ids) - 1
            labels = [ids[i] if i >= plen else -100 for i in range(len(ids))]
            self.data.append({"input_ids": ids, "attention_mask": full["attention_mask"], "labels": labels})
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        return {k: torch.tensor(v) for k, v in d.items()}

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
ds = SFT(load(train_file[0]), tok)
print("train samples:", len(ds))

In [ ]:
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model.enable_input_require_grads()

lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    push_to_hub=False,
)
coll = DataCollatorForSeq2Seq(tokenizer=tok, model=model, padding=True, label_pad_token_id=-100)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=coll)
trainer.train()

In [ ]:
# 保存 LoRA adapter + 合并为完整模型
adapter_dir = OUT_DIR + "/adapter"
model.save_pretrained(adapter_dir); tok.save_pretrained(adapter_dir)
print("adapter saved:", adapter_dir)

merged = PeftModel.from_pretrained(model, adapter_dir).merge_and_unload()
merged_dir = OUT_DIR + "/merged"
merged.save_pretrained(merged_dir, safe_serialization=True); tok.save_pretrained(merged_dir)
print("merged saved:", merged_dir)

In [ ]:
# 打包下载
!tar -czf /kaggle/working/qwen3-ugov-merged.tar.gz -C /kaggle/working/ugov-qwen merged
!ls -lh /kaggle/working/qwen3-ugov-merged.tar.gz
print("下载文件: /kaggle/working/qwen3-ugov-merged.tar.gz   (右侧 Output 面板)")

In [ ]:
# 快速自测（可选）：用微调后模型生成一条回复
import transformers
gen = AutoModelForCausalLM.from_pretrained(merged_dir, torch_dtype=torch.bfloat16, device_map="auto")
msgs = [{"role":"user","content":"我们小区楼下烧烤摊每晚油烟直排、音响扰民，还占道经营，多次反映没人管，麻烦尽快处理。"}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
ids = tok(text, return_tensors="pt").to("cuda")
out = gen.generate(**ids, max_new_tokens=180, do_sample=True, temperature=0.3)
print(tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True))

## 下载后接回项目（Ollama）

1. 在 Output 下载 `qwen3-ugov-merged.tar.gz`，解压得到 `merged/`（HF 结构）。
2. 用 llama.cpp 转 GGUF 并导入：`python convert_hf_to_gguf.py merged -o qwen3-ugov.gguf` → `ollama create qwen3-ugov -f Modelfile`。
3. 项目 `.env` 设 `LLM_MODE=local`、`LOCAL_MODEL=qwen3-ugov`。

详见 `docs/KAGGLE_FINETUNE.md`。